<div style='background:#1a1a2e;color:#eee;padding:18px;border-radius:8px;margin-bottom:12px'>
<h2 style='margin:0'>🍎 EDUCATOR VERSION — Chapter 13: Probability</h2>
<p style='margin:6px 0 0'>This notebook contains answer keys, teaching notes, and discussion prompts. Students should use the standard <strong>Chapter_13.ipynb</strong>.</p>
</div>


# Chapter 13: Probability in Structural Engineering
## What Are the Chances This Structure Survives Its Design Life?

**Learning Objectives:**
- Define probability in terms of long-run frequency and apply it to structural load events
- Calculate the probability that an extreme load occurs at least once over a structure's design life
- Explain what a '100-year load' actually means — and what it does *not* mean
- Connect structural reliability to the everyday idea of probability


<center>
<img src='https://upload.wikimedia.org/wikipedia/commons/thumb/9/96/I-35W_Saint_Anthony_Falls_Bridge.jpg/1280px-I-35W_Saint_Anthony_Falls_Bridge.jpg' width='700' />

<em>The St. Anthony Falls Bridge, Minneapolis — replacement for the I-35W bridge that collapsed in 2007. Before opening in 2008, MnDOT placed 2 million pounds of test loads on the bridge and measured deflections at dozens of locations. This is proof load testing in action. (Wikimedia Commons — public domain.)</em>
</center>

---


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Context

**Curriculum connections:**
- *Statistics class*: Long-run frequency interpretation of probability; geometric distribution (time until first event); complement rule; independent events
- *Math class*: Exponential functions; limit of (1 − 1/n)ⁿ → 1/e as n → ∞
- *Physics class*: Measurement uncertainty over time; return period as engineering design parameter

**Prerequisites:** Students should be comfortable with probability as long-run frequency and understand the multiplication rule for independent events.

**Estimated time:** 45–55 minutes. The concept of return period takes most students 10–15 minutes to internalize; budget time for the '100-year flood' misconception discussion.

**Pedagogical note:** The '100-year flood' misconception is nearly universal among students (and the general public). The most effective approach is to ask students *before* revealing the formula: 'If a flood happens once every 100 years on average, and one just happened this year, how long until the next one?' Most will say '100 years.' The simulation widget then shows them how wrong that intuition is — floods cluster and the gaps vary wildly.
</div>


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
np.random.seed(13)
print('Setup complete.')


## 13.1  Probability and Structural Design

Every structure faces loads that cannot be known in advance — gusts of wind, earthquakes, heavy snowstorms, overloaded trucks. Structural engineers use **probability** to describe how likely these events are and to choose design loads that are appropriately conservative.

The fundamental question is: **over the life of this structure, what is the probability it will be subjected to a load exceeding its design capacity?**

Hibbeler §1.3 notes that design loads are specified by codes like ASCE 7-10 and are determined from *historical observation of loading events*. Those code values are not the absolute maximum load ever recorded — they are loads with a specified **annual probability of exceedance**. *(Note: Hibbeler references ASCE 7-10 for load values but does not derive return periods directly — that derivation lives in ASCE 7-10 itself. This chapter uses return periods as context for the probability concepts, not as textbook content students are expected to reproduce.)*

### The Return Period

A **return period** (also called a *recurrence interval*) of T years means:

$$P(\text{load exceeded in any one year}) = \frac{1}{T}$$

- A **50-year wind** has a 1/50 = **2% annual probability** of being exceeded
- A **100-year flood** has a 1/100 = **1% annual probability** of being exceeded
- A **475-year earthquake** has a 1/475 ≈ **0.21% annual probability** of being exceeded

⚠️ **Common misconception:** A "100-year flood" does NOT mean it happens once every 100 years like clockwork. It means there is a 1% chance of it happening *in any given year* — which means it could happen two years in a row, or not at all for 200 years.


## 🔬 Interactive Experiment 1: Design Life vs. Return Period

If there is a 1% chance of a load being exceeded in any single year, what is the probability it gets exceeded *at least once* over the structure's full design life?

If each year is independent, the probability of the load being exceeded at least once in n years is:

$$P(\text{at least one exceedance in } n \text{ years}) = 1 - \left(1 - \frac{1}{T}\right)^n$$

Use the sliders to explore how design life and return period interact.


In [ ]:
def _lifetime_prob(return_period, design_life):
    p_annual = 1.0 / return_period
    p_lifetime = 1 - (1 - p_annual) ** design_life

    years = np.arange(1, 151)
    curves = [
        (50,   'steelblue',  '50-yr return (wind)'),
        (100,  'darkorange', '100-yr return (flood)'),
        (475,  'firebrick',  '475-yr return (earthquake, 10% in 50yr)'),
        (2475, 'purple',     '2475-yr return (earthquake, 2% in 50yr)'),
    ]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for T, color, label in curves:
        probs = 1 - (1 - 1/T)**years
        ax1.plot(years, probs*100, color=color, lw=2, label=label)
    ax1.axvline(design_life, color='black', lw=1.5, ls='--', label=f'Your design life: {design_life} yr')
    ax1.set_xlabel('Design Life (years)', fontsize=12)
    ax1.set_ylabel('Probability of At Least One Exceedance (%)', fontsize=12)
    ax1.set_title('Lifetime Exceedance Probability by Return Period', fontsize=13)
    ax1.legend(fontsize=9)
    ax1.set_ylim(0, 100)

    # Right: bar for selected values
    selected_probs = []
    labels2 = []
    for T, color, label in curves:
        selected_probs.append((1-(1-1/T)**design_life)*100)
        labels2.append(label.split('(')[0].strip())
    bars = ax2.bar(labels2, selected_probs, color=[c[1] for c in curves], edgecolor='white')
    ax2.axhline(10, color='gray', ls=':', lw=1.5, label='10% threshold')
    for bar, val in zip(bars, selected_probs):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}%',
            ha='center', fontsize=10)
    ax2.set_ylabel('Probability of Exceedance (%)', fontsize=12)
    ax2.set_title(f'Over a {design_life}-Year Design Life', fontsize=13)
    ax2.set_ylim(0, 110)
    ax2.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f'\nYour selection: T = {return_period} years, n = {design_life} years')
    print(f'P(at least one exceedance) = 1 - (1 - 1/{return_period})^{design_life} = {p_lifetime*100:.1f}%')

w_T = widgets.IntSlider(value=100, min=10, max=2500, step=25,
    description='Return period T (yr):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
w_n = widgets.IntSlider(value=50, min=5, max=150, step=5,
    description='Design life n (yr):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out1 = widgets.interactive_output(_lifetime_prob, {'return_period': w_T, 'design_life': w_n})
display(widgets.VBox([w_T, w_n, out1]))


---
## ⚠️  Real-World Case: How AASHTO Chooses a 75-Year Return Period for Bridge Live Loads

The **AASHTO LRFD Bridge Design Specifications** — the primary code for highway bridges in the United States — specify that bridge live loads shall be based on a **75-year return period**. Why 75 years? Because that is the standard design life for a new highway bridge.

Here is the probability reasoning behind that choice:

The AASHTO HL-93 design truck represents the heaviest vehicle load with approximately a **1-in-75 annual probability** of being equaled or exceeded on a given bridge. Over a 75-year design life:

$$P(\\text{design load exceeded at least once}) = 1 - \\left(1 - \\frac{1}{75}\\right)^{75} \\approx 63\\%$$

In other words, the AASHTO live load is **not** a load that will never be exceeded. It is a load with a 63% chance of being exceeded at least once over the bridge's life. Engineers accept this because:

1. **Load factors** in AASHTO LRFD (1.75 × live load for the strength limit state) add a margin above the nominal design load
2. **Resistance factors** (φ ≈ 0.9 for bending) ensure the beam's actual strength exceeds the factored demand
3. Together, these keep the true probability of structural failure far below 63%

The 75-year return period is not a guess — it is the answer to a specific probability question: *what load level, combined with appropriate safety factors, keeps bridge failure probability acceptably low over a standard design life?*

> *Return periods in structural codes are chosen by working backward from acceptable risk — not forward from historical records alone.*


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Note — AASHTO Bridge Design Case Study

The AASHTO 75-year return period is one of the clearest examples of *engineering code as probability statement*. Students sometimes think codes are arbitrary rules passed down by authority — this case study shows the opposite: a specific probability calculation drove the specification.

**The math behind the code:**
- Design life = 75 years (standard highway bridge service life)
- Target: the nominal design load should be the load with a 1-in-75 annual probability
- P(exceeded at least once) = 1 − (1 − 1/75)^75 ≈ 63%

**Why accept 63% probability of exceedance?** Because the bridge is not designed to fail when the nominal load is exceeded — it is designed to *not fail* when the factored load (1.75 × nominal for truck live load) is reached. The probability of the factored load being exceeded in 75 years is much smaller.

**Connection to the memoryless property:** The geometric distribution that describes 'time until first exceedance' is memoryless — a bridge that has already served 30 years without seeing a 75-year event does not have a reduced probability of seeing one in the next year. Each year is an independent trial. This is counterintuitive but important.
</div>


## 🔬 Interactive Experiment 2: Simulating a Bridge's Load History

The simulation below generates a random load history for a bridge over its design life. Each year, there is a probability p that an extreme load event (flood, major storm, earthquake) occurs. Some years it happens. Some it does not. The randomness is real.

Run the simulation many times and observe: even with the same probability, no two lifetime histories look the same — but the long-run statistics are stable.


In [ ]:
def _simulate_lifetime(annual_prob_pct, design_life, n_simulations):
    p = annual_prob_pct / 100
    exceedances_per_life = []
    for _ in range(n_simulations):
        events = np.random.random(design_life) < p
        exceedances_per_life.append(events.sum())

    theoretical = 1 - (1-p)**design_life
    simulated   = np.mean(np.array(exceedances_per_life) >= 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # One example lifetime timeline
    example = np.random.random(design_life) < p
    years_hit = np.where(example)[0] + 1
    ax1.bar(range(1, design_life+1), np.ones(design_life)*0.3, color='lightgray', width=1)
    if len(years_hit):
        ax1.bar(years_hit, np.ones(len(years_hit)), color='firebrick', width=1,
            label=f'Exceedance ({len(years_hit)} events)')
    ax1.set_xlabel('Year', fontsize=12)
    ax1.set_yticks([])
    ax1.set_title(f'One Simulated {design_life}-Year Lifetime', fontsize=13)
    ax1.legend(fontsize=10)

    # Distribution of exceedance count across simulations
    max_count = max(exceedances_per_life) + 1
    ax2.hist(exceedances_per_life, bins=range(0, max_count+1), color='steelblue',
        edgecolor='white', align='left', density=True)
    ax2.set_xlabel('Number of Exceedances in Design Life', fontsize=12)
    ax2.set_ylabel('Fraction of Simulated Lifetimes', fontsize=12)
    ax2.set_title(f'{n_simulations} Simulated Lifetimes', fontsize=13)
    ax2.annotate(
        f'Theoretical P(≥1): {theoretical*100:.1f}%\nSimulated P(≥1):    {simulated*100:.1f}%',
        xy=(0.55, 0.82), xycoords='axes fraction', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    plt.tight_layout()
    plt.show()

w_p  = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
    description='Annual P (%):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
w_n  = widgets.IntSlider(value=50, min=10, max=150, step=5,
    description='Design life (yr):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
w_ns = widgets.IntSlider(value=500, min=100, max=2000, step=100,
    description='Simulations:', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out2 = widgets.interactive_output(_simulate_lifetime,
    {'annual_prob_pct': w_p, 'design_life': w_n, 'n_simulations': w_ns})
display(widgets.VBox([w_p, w_n, w_ns, out2]))


---
## 📋  Chapter 13 Review

| Term | Meaning |
|------|--------|
| **Probability** | Long-run relative frequency of an event |
| **Annual exceedance probability** | Chance a load is exceeded in any single year: p = 1/T |
| **Return period T** | Average number of years between exceedances; T = 1/p |
| **Design life** | Number of years a structure is expected to serve |
| **Lifetime exceedance probability** | P(at least one exceedance in n years) = 1 − (1−p)ⁿ |

**The Big Idea:** Structural engineers do not design for the worst load that has ever been observed — they design for the worst load with a specified probability of being exceeded over a structure's lifetime. Understanding that probability requires the same rules you learn in statistics: probability of complement, independent events, and the multiplication rule. A '100-year event' is not a prediction. It is a probability statement — and over a 50-year bridge life, there is nearly a 40% chance of experiencing one.


<div style='background:#d1ecf1;padding:15px;border-left:5px solid #17a2b8;margin:10px 0'>

### 💬 Discussion Prompts

**1. The '500-year flood' (warm-up):** In 2015, South Carolina experienced what officials called a '1000-year rain event.' The previous year, the same area had a '500-year flood.' Is this statistically impossible? *(No — each year is independent. Rare events can cluster. The labels reflect annual probability, not spacing.)*

**2. Acceptable risk (pair discussion):** ASCE 7-10 uses a 2%-in-50-years earthquake (2475-year return period) for the design earthquake in the US. Is a 2% chance of being shaken that hard over 50 years 'acceptable'? Who decides what level of risk is acceptable? *(Guide toward: engineers, building codes, insurance actuaries, elected officials, and ultimately society set these thresholds — it's partly technical, partly political.)*

**3. Design life choices (small group):** A nuclear waste storage facility is designed for 10,000 years. A temporary construction bridge is designed for 5 years. How does design life change what return period you should use? Draw a sketch of P(exceedance) vs. design life for a few return periods and discuss what the curves imply.

**4. Extension (homework):** Look up the '100-year floodplain' designation used by FEMA for flood insurance purposes. What annual probability does this represent? If a house is in the 100-year floodplain and you live there for 30 years, what is the probability your house floods at least once? Show the calculation.
</div>
